# Building Nutrient Gradient Sweep Function for Infection Medium Tuning

```
see Obsidian Notes:
08.07.26, 08.08.26, 08.10.26, 08.11.26

continued from:
/home/gmvaz/2026_GEMs/stm_model_test/0728_0731_medium_concentration.ipynb

tmux: gems
conda environment: gems
```

# Generate Constrained Models of the STM only condition and the Mixed STM condition

In [1]:
# Set the gurobi solver 

import os
from cobra.core.configuration import Configuration

os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

# Read in the gene weights CSV files

import pandas as pd

stm_threshold_10_90 = pd.read_csv("../imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("../imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("../imat_prep_test/stm_threshold_25_75.csv")

mixed_stm_threshold_10_90 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("../imat_prep_test/mixed_stm_threshold_25_75.csv")

# Make a python set of each gene weights CSV

stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

mixed_stm_set_10_90 = set(mixed_stm_threshold_10_90["genes"].astype(str).str.strip())
mixed_stm_set_15_85 = set(mixed_stm_threshold_15_85["genes"].astype(str).str.strip())
mixed_stm_set_25_75 = set(mixed_stm_threshold_25_75["genes"].astype(str).str.strip())

# QC: Check the count of genes

print("STM1090 gene count",len(stm_set_10_90), "|", "STM1585 gene count", len(stm_set_15_85), "|", "STM2575 gene count", len(stm_set_25_75))
print("Mixed STM1090 gene count",len(mixed_stm_set_10_90), "|", "Mixed STM1585 gene count", len(mixed_stm_set_15_85), "|", "Mixed STM2575 gene count", len(mixed_stm_set_25_75))

STM1090 gene count 4671 | STM1585 gene count 4671 | STM2575 gene count 4671
Mixed STM1090 gene count 4671 | Mixed STM1585 gene count 4671 | Mixed STM2575 gene count 4671


In [2]:
# Load STM_V1_0

from cobra.io import load_model, load_json_model

control_model = load_model("STM_v1_0")

# Save all genes in the control model in a set

model_genes = set([g.id for g in control_model.genes])

# QC: Check the count of genes
print("STM_V1_0 Control Model has", len(model_genes), "total genes")

# Subset the gene weights lists for each sample to keep only the genes also found in the control model

stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

mixed_stm1090_model = mixed_stm_threshold_10_90[mixed_stm_threshold_10_90["genes"].isin(model_genes)]
mixed_stm1585_model = mixed_stm_threshold_15_85[mixed_stm_threshold_15_85["genes"].isin(model_genes)]
mixed_stm2575_model = mixed_stm_threshold_25_75[mixed_stm_threshold_25_75["genes"].isin(model_genes)]

# Align the gene weights lists to the control model genes list and assign missing genes a weight of 0

subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

subset_weights_mixedstm1090 = mixed_stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm1585 = mixed_stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_mixedstm2575 = mixed_stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)


# QC: Confirm all samples have the same amount of gene weights

# 1. Make a list of the subset gene weights
subset_gene_weights = [
    subset_weights_stm1090,
    subset_weights_stm1585,
    subset_weights_stm2575,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm1090,
    subset_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
gene_counts = [len(weights) for weights in subset_gene_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(gene_counts)) == 1:
    print(f"All models have {gene_counts[0]} genes")
else:
    print("Gene counts differ between models")
    print(gene_counts)

# Convert gene weights to reaction weights with iMAT

from imatpy.parse_gpr import gene_to_rxn_weights

rxn_weights_stm1090 = gene_to_rxn_weights(control_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(control_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(control_model, subset_weights_stm2575)

rxn_weights_mixedstm1090 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1090)
rxn_weights_mixedstm1585 = gene_to_rxn_weights(control_model, subset_weights_mixedstm1585)
rxn_weights_mixedstm2575 = gene_to_rxn_weights(control_model, subset_weights_mixedstm2575)

# QC: Confirm all samples have the same amount of reaction weights

# 1. Make a list of the reaction weights
reaction_weights = [
    rxn_weights_stm1090,
    rxn_weights_stm1585,
    rxn_weights_stm2575,
    rxn_weights_mixedstm1090,
    rxn_weights_mixedstm1585,
    rxn_weights_mixedstm2575]

# 2. Make a list of how many genes are in each list
reactions_counts = [len(weights) for weights in reaction_weights]

# 3. Compare how many genes are in each list and print the total if they are all the same
if len(set(reactions_counts)) == 1:
    print(f"All models have {reactions_counts[0]} reactions")
else:
    print("Reaction counts differ between models")
    print(reactions_counts)

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu
STM_V1_0 Control Model has 1271 total genes
All models have 1271 genes
All models have 2545 reactions


In [3]:
# Generate constrained models with iMAT by integrating the reaction weights

from imatpy.model_creation import generate_model

stm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

mixedstm1090_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1090, method = "imat_restrictions")
mixedstm1585_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm1585, method = "imat_restrictions")
mixedstm2575_model = generate_model(model = control_model, rxn_weights = rxn_weights_mixedstm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmp2_k67_rf.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp8kcuv854.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmps1m2ai91.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp6p6gka67.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmp4735cf74.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpxzcy_721.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [4]:
stm1090_model

Name,STM_v1_0
Memory address,dce15577350
Number of metabolites,1802
Number of reactions,2545
Number of genes,1271
Number of groups,0
Objective expression,1.0*BIOMASS_iRR1083_metals - 1.0*BIOMASS_iRR1083_metals_reverse_559d4
Compartments,"cytosol, periplasm, extracellular space"


# Make the `original_models` dicitonary of models

In [5]:
# Make a dictionary of the models 

original_models = {
    "stm_1090": stm1090_model,
    "stm_1585": stm1585_model,
    "stm_2575": stm2575_model,
    "mixed_stm1090": mixedstm1090_model,
    "mixed_stm1585": mixedstm1585_model,
    "mixed_stm2575": mixedstm2575_model
}

# Make a dictionary with copies of each original model

fba_models = {
    model_name: model.copy()
    for model_name, model in original_models.items()
}

Read LP format model from file /tmp/tmp0cypq0uv.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmproxnvlvw.lp
Reading time = 0.02 seconds
: 3201 rows, 6460 columns, 25176 nonzeros
Read LP format model from file /tmp/tmp9ljd152b.lp
Reading time = 0.03 seconds
: 3949 rows, 7128 columns, 28088 nonzeros
Read LP format model from file /tmp/tmp3jsy9seq.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpbr5vw5oc.lp
Reading time = 0.02 seconds
: 3179 rows, 6432 columns, 25082 nonzeros
Read LP format model from file /tmp/tmpq3ze05a2.lp
Reading time = 0.02 seconds
: 3901 rows, 7068 columns, 27884 nonzeros


In [6]:
fba_models

{'stm_1090': <Model STM_v1_0 at 0xdcd7f63ec90>,
 'stm_1585': <Model STM_v1_0 at 0xdcd7f486050>,
 'stm_2575': <Model STM_v1_0 at 0xdcd81cb73d0>,
 'mixed_stm1090': <Model STM_v1_0 at 0xdcd7adf68d0>,
 'mixed_stm1585': <Model STM_v1_0 at 0xdcd7676c9d0>,
 'mixed_stm2575': <Model STM_v1_0 at 0xdcd7550ded0>}

# Define the Infection Medium Conditions

In [7]:
# Make a copy of the default medium from any of the models (stm1090 used)

model_copy = fba_models.get("stm_1090")
default_medium = model_copy.medium

# Make 7 identical infection mediums with oxygen ranging from 0 - 3 at 0.5 increments

A_infection_medium = dict(default_medium)
A_infection_medium.update({
    "EX_o2_e": 0.0, # removing from default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

B_infection_medium = dict(default_medium)
B_infection_medium.update({
    "EX_o2_e": 0.5, # changing in default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

C_infection_medium = dict(default_medium)
C_infection_medium.update({
    "EX_o2_e": 1.0, # changing in the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

D_infection_medium = dict(default_medium)
D_infection_medium.update({
    "EX_o2_e": 1.5, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

E_infection_medium = dict(default_medium)
E_infection_medium.update({
    "EX_o2_e": 2.0, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

F_infection_medium = dict(default_medium)
F_infection_medium.update({
    "EX_o2_e": 2.5, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

G_infection_medium = dict(default_medium)
G_infection_medium.update({
    "EX_o2_e": 3.0, # changing from the default medium
    "EX_tet_e": 4.0,
    "EX_no3_e": 0.5,
    "EX_etha_e": 1.0,
    "EX_12ppd__R_e": 0.1,
    "EX_12ppd__S_e": 0.1,
    "EX_lac__L_e": 10.0,
    "EX_for_e": 4.0,
    "EX_succ_e": 3.0,
    "EX_asp__L_e": 5.0,
    "EX_mal__L_e": 0.2,
    "EX_galct__D_e": 10.0,
    "EX_glcr_e": 10.0,
    "EX_pyr_e": 10.0,
    "EX_arab__L_e": 13.3,
    "EX_but_e": 10.0,
    "EX_ac_e": 50.0,
    "EX_ppa_e": 5.0,
    "EX_glc__D_e": 0.0 # removing from default medium
})

# Make `base_medias` - dicitonary of the infection medium's

In [8]:
base_medias = {
	"A_infection_medium": A_infection_medium,
	"B_infection_medium": B_infection_medium,
	"C_infection_medium": C_infection_medium,
	"D_infection_medium": D_infection_medium,
	"E_infection_medium": E_infection_medium,
	"F_infection_medium": F_infection_medium,
	"G_infection_medium": G_infection_medium
}

# Make the `nutrient_rxn_list` -  list of reaction_ID's who's value will be changed

In [9]:
nutrient_rxn_list = list(A_infection_medium.keys())
nutrient_rxn_list.remove('EX_o2_e')
nutrient_rxn_list

['EX_ca2_e',
 'EX_cbl1_e',
 'EX_cl_e',
 'EX_co2_e',
 'EX_cobalt2_e',
 'EX_cu2_e',
 'EX_mg2_e',
 'EX_fe2_e',
 'EX_fe3_e',
 'EX_mn2_e',
 'EX_mobd_e',
 'EX_na1_e',
 'EX_nh4_e',
 'EX_pi_e',
 'EX_zn2_e',
 'EX_so4_e',
 'EX_glc__D_e',
 'EX_tungs_e',
 'EX_h2o_e',
 'EX_h_e',
 'EX_k_e',
 'EX_tet_e',
 'EX_no3_e',
 'EX_etha_e',
 'EX_12ppd__R_e',
 'EX_12ppd__S_e',
 'EX_lac__L_e',
 'EX_for_e',
 'EX_succ_e',
 'EX_asp__L_e',
 'EX_mal__L_e',
 'EX_galct__D_e',
 'EX_glcr_e',
 'EX_pyr_e',
 'EX_arab__L_e',
 'EX_but_e',
 'EX_ac_e',
 'EX_ppa_e']

# Make the `nutrient_step_count_list`

In [10]:
import numpy as np

nutrient_step_count = np.logspace(0.5, 7, num = 15, base = 2.0)

nutrient_step_count_array = np.insert(nutrient_step_count, 0, 0)

nutrient_step_count_list = nutrient_step_count_array.tolist()

nutrient_step_count_list

[0.0,
 1.4142135623730951,
 1.9510972841051464,
 2.6918003852647123,
 3.7136996566701033,
 5.12354675905709,
 7.06861992598028,
 9.752109233635803,
 13.454342644059432,
 18.562070178562607,
 25.608865362590898,
 35.330864437562006,
 48.74366607935861,
 67.24842487380603,
 92.77822149538744,
 128.0]

# Define `run_medium_sweep()`


def run_medium_sweep(*, model_name, medium_name, nutrient_rxn, nutrient_step_count): 
    
    results = [ ]
    
    for nutrient_value in nutrient_step_count:
        
        model = fba_models[model_name].copy()
        medium = base_medias[medium_name].copy()
        medium[nutrient_rxn] = float(nutrient_value)
        
        try:
            with model:
                model.medium = medium
                solution = model.optimize()

                if solution.status == "optimal": 
                    objective_value = solution.objective_value
                else:
                    objective_value = np.nan # NaN - Not a Number

                results.append({
                    "model": model_name,
                    "medium_condition": medium_name,
                    "nutrient_rxn": nutrient_rxn,
                    "nutrient_rxn_value": float(nutrient_value),
                    "status": solution.status,
                    "objective_value": objective_value
                })
                
        except Exception as error:
            results.append({
                "model": model_name,
                "medium_condition": medium_name,
                "nutrient_rxn": nutrient_rxn,
                "nutrient":float(nutrient_value),
                "status": "failed",
                "objective_value": np.nan,
                "error": str(error)
            })

    results_df = pd.DataFrame(results)

    return results_df

# Run `run_medium_sweep()` on Stm1090_Ca2 with Medium's A - G

In [12]:
stm1090_ca2_results = run_medium_sweep(model_name = "stm_1090", medium_name = "A_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)

Read LP format model from file /tmp/tmpkm37anc1.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp_dekrso3.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp63ohe6_t.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp2wdu8bk4.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp98su_kr6.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp7v5309c2.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpqapldtoz.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpa8kuogeq.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [14]:
stm1090_ca2_results.to_csv("nutrient_sweep_081226/stm1090_ca2_A_medium_results.csv")

stm1090_ca2_results[:10]

,model,medium_condition,nutrient_rxn,nutrient_rxn_value,status,objective_value
0,stm_1090,A_infection_medium,EX_ca2_e,0.000000,optimal,0.000000
1,stm_1090,A_infection_medium,EX_ca2_e,1.414214,optimal,0.844452
2,stm_1090,A_infection_medium,EX_ca2_e,1.951097,optimal,0.844452
3,stm_1090,A_infection_medium,EX_ca2_e,2.691800,optimal,0.844403
4,stm_1090,A_infection_medium,EX_ca2_e,3.713700,optimal,0.844452
5,stm_1090,A_infection_medium,EX_ca2_e,5.123547,optimal,0.844452
6,stm_1090,A_infection_medium,EX_ca2_e,7.068620,optimal,0.844452
7,stm_1090,A_infection_medium,EX_ca2_e,9.752109,optimal,0.844452
8,stm_1090,A_infection_medium,EX_ca2_e,13.454343,optimal,0.844452
9,stm_1090,A_infection_medium,EX_ca2_e,18.562070,optimal,0.844452


In [37]:
import pandas
isinstance(stm1090_ca2_results, pandas.core.frame.DataFrame)

True

In [15]:
stm1090_ca2_B_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "B_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_B_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_B_medium_results.csv")

Read LP format model from file /tmp/tmp46ealfu_.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp8ulcry_a.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp_i2o6na6.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp7f9v6r4z.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpglp60z_r.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpr226zuyy.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpmrpvmr6h.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptm1c8wwz.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [16]:
stm1090_ca2_C_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "C_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_C_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_C_medium_results.csv")

Read LP format model from file /tmp/tmp_7jdmscl.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpkbmirkfs.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp4n7vimx8.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmppn7630ni.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp7uwm_x7d.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptnosnbge.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp2_s9ft96.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp8n3eoawa.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [17]:
stm1090_ca2_D_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "D_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_D_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_D_medium_results.csv")

Read LP format model from file /tmp/tmpxgf4ou_y.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpukimqceo.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpi37pq021.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp261_98mg.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmps3zv674z.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp8mgxkky7.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp6en475l4.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpk3es1qbf.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [18]:
stm1090_ca2_E_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "E_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_E_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_E_medium_results.csv")

Read LP format model from file /tmp/tmpq2_8cpg6.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp14dgv9g4.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpbxlhbsic.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpkv2vg_kz.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpsb24u6vw.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp5hfmndm8.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp9dmbrz1a.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmphw50fdc0.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [19]:
stm1090_ca2_F_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "F_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_F_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_F_medium_results.csv")


Read LP format model from file /tmp/tmpth_k7svd.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpmj511556.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptzls_ydx.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpkkw9048m.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpqhgutm2e.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpulyjq41q.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpzba291hi.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpyekbfn4b.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

In [20]:
stm1090_ca2_G_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "G_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
stm1090_ca2_G_medium_results.to_csv("nutrient_sweep_081226/stm1090_ca2_G_medium_results.csv")

Read LP format model from file /tmp/tmp2c1arxfk.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp4fi66rh2.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmp1p5x6xuq.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptaw7ub9_.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmptxxd3jel.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpomwzwiye.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpqu5vn0lt.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpne6mw6fe.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /

# Run `run_medium_sweep()` on Mixed_Stm1090_Ca2 with Medium's A - G

In [21]:
mixed_stm1090_ca2_A_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "A_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_A_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_A_medium_results.csv")

Read LP format model from file /tmp/tmpcm4faalz.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpm0vjltb9.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpor866jqt.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpquv3uqsl.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmppn6puzeq.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp8w351sgr.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpmosg8_p1.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpmfjpgthw.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [22]:
mixed_stm1090_ca2_B_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "B_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_B_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_B_medium_results.csv")

Read LP format model from file /tmp/tmp1i1nnn01.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp02kclu7v.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpxtts8wb2.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp21aw5bb8.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpdtdzyzoh.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpoouh06kh.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmptpjw1o53.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpt57m0t2r.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [23]:
mixed_stm1090_ca2_C_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "C_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_C_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_C_medium_results.csv")

Read LP format model from file /tmp/tmpodyrf8c6.lp
Reading time = 0.03 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6egfco35.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpcu5tr9ge.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpcgdkkouo.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp2350quar.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp3nhusu39.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpjnztmgpi.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpkizvzofk.lp
Reading time = 0.03 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [24]:
mixed_stm1090_ca2_D_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "D_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_D_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_D_medium_results.csv")

Read LP format model from file /tmp/tmpioma67jv.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpgfc7vtzg.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpjj8fqfox.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp9m52vhw3.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp9pvgc9l3.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6lstktf5.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6edy74wi.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmptb0xh17w.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [25]:
mixed_stm1090_ca2_E_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "E_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_E_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_E_medium_results.csv")

Read LP format model from file /tmp/tmpejk4gmoj.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpfq5v7nef.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp0f7rhlim.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmplgcfk349.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpeify3pmb.lp
Reading time = 0.03 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpbbvwz32k.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpur7fli7e.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpahs_p303.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [29]:
mixed_stm1090_ca2_F_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "F_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_F_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_F_medium_results.csv")

Read LP format model from file /tmp/tmpay3jllv7.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp2vsonu43.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpq6g1oh10.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6r_gejh2.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpduudmype.lp
Reading time = 0.03 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpchecpmlv.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp0bvm9d1t.lp
Reading time = 0.03 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpa5n36xxs.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

In [30]:
mixed_stm1090_ca2_G_medium_results = run_medium_sweep(model_name = "mixed_stm1090", medium_name = "G_infection_medium", nutrient_rxn = "EX_ca2_e", nutrient_step_count = nutrient_step_count_list)
mixed_stm1090_ca2_G_medium_results.to_csv("nutrient_sweep_081226/mixed_stm1090_ca2_G_medium_results.csv")

Read LP format model from file /tmp/tmpooyjp8pz.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpnmbmuc_1.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6xjxhh6y.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp6haq11m0.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpqkfl4z06.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpdxu_jtjo.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmp7e9_nen1.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /tmp/tmpbn5qld79.lp
Reading time = 0.02 seconds
: 2901 rows, 6178 columns, 23994 nonzeros
Read LP format model from file /

KeyboardInterrupt: 

# `run_medium_sweep()` with `model.slim_optimize()` instead of `model.optimize`

In [38]:
def run_medium_sweep(*, model_name, medium_name, nutrient_rxn, nutrient_step_count): 
    
    results = [ ]
    
    for nutrient_value in nutrient_step_count:
        
        model = fba_models[model_name].copy()
        medium = base_medias[medium_name].copy()
        medium[nutrient_rxn] = float(nutrient_value)
        
        try:
            with model:
                model.medium = medium
                objective_value = model.slim_optimize()

                if isinstance(objective_value, float) == True:
                        status = "optimal"
                else:
                    status = "infeasible"

                results.append({
                    "model": model_name,
                    "medium_condition": medium_name,
                    "nutrient_rxn": nutrient_rxn,
                    "nutrient_rxn_value": float(nutrient_value),
                    "status": status,
                    "objective_value": objective_value
                })
                
        except Exception as error:
            #raise
            results.append({
                "model": model_name,
                "medium_condition": medium_name,
                "nutrient_rxn": nutrient_rxn,
                "nutrient":float(nutrient_value),
                "status": "failed",
                "objective_value": np.nan,
                "error": str(error)
            })

    results_df = pd.DataFrame(results)

    return results_df

In [39]:
stm1090_cbl1_A_medium_results = run_medium_sweep(model_name = "stm_1090", medium_name = "A_infection_medium", nutrient_rxn = "EX_cbl1_e", nutrient_step_count = nutrient_step_count_list)
stm1090_cbl1_A_medium_results.to_csv("nutrient_sweep_081226/stm1090_cbl1_A_medium_results.csv")

Read LP format model from file /tmp/tmpdqh4le8p.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpxdap5ad5.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpjlkynq24.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpdgoysh9t.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpelp80rqv.lp
Reading time = 0.03 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpwj0qszjf.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmpuy_w4apv.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /tmp/tmprk4pf2f3.lp
Reading time = 0.02 seconds
: 2947 rows, 6220 columns, 24174 nonzeros
Read LP format model from file /